# Predictive Maintenance - RUL Prediction

In this project i am trying to predict how many more cycles a turbofan engine can run before it breaks down. This is called RUL (Remaining Useful Life).

Dataset i am using is NASA C-MAPSS FD001 dataset. It has sensor readings from multiple engines and we know when each engine failed.

So basically train data has engines running till failure, test data has engines stopped before failure and we need to predict how many cycles are left.

## Step 1 - Importing all required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import pickle

## Step 2 - Loading the dataset

The txt files dont have column names so i have to give them manually.
First 2 columns are engine id and cycle number, then 3 operational settings, then 21 sensor readings.

In [2]:
# making column names manually
cols = ['unit_id', 'cycle', 'op1', 'op2', 'op3']
for i in range(1, 22):
    cols.append('sensor_' + str(i))

print(cols)

['unit_id', 'cycle', 'op1', 'op2', 'op3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [3]:
train_df = pd.read_csv('Data/train_FD001.txt', sep='\s+', header=None, names=cols)
print(train_df.shape)
train_df.head()

(20631, 26)


,unit_id,cycle,op1,op2,op3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [4]:
test_df = pd.read_csv('Data/test_FD001.txt', sep='\s+', header=None, names=cols)
print(test_df.shape)
test_df.head()

(13096, 26)


,unit_id,cycle,op1,op2,op3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [5]:
# this file has actual RUL values for test engines
rul_df = pd.read_csv('Data/RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])
rul_df.head()

,RUL
0,112
1,98
2,69
3,82
4,91


## Step 3 - Understanding the data

In [6]:
print('rows:', train_df.shape[0])
print('columns:', train_df.shape[1])
print('total engines in train:', train_df['unit_id'].nunique())

rows: 20631
columns: 26
total engines in train: 100


In [7]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   unit_id    20631 non-null  int64  
 1   cycle      20631 non-null  int64  
 2   op1        20631 non-null  float64
 3   op2        20631 non-null  float64
 4   op3        20631 non-null  float64
 5   sensor_1   20631 non-null  float64
 6   sensor_2   20631 non-null  float64
 7   sensor_3   20631 non-null  float64
 8   sensor_4   20631 non-null  float64
 9   sensor_5   20631 non-null  float64
 10  sensor_6   20631 non-null  float64
 11  sensor_7   20631 non-null  float64
 12  sensor_8   20631 non-null  float64
 13  sensor_9   20631 non-null  float64
 14  sensor_10  20631 non-null  float64
 15  sensor_11  20631 non-null  float64
 16  sensor_12  20631 non-null  float64
 17  sensor_13  20631 non-null  float64
 18  sensor_14  20631 non-null  float64
 19  sensor_15  20631 non-null  float64
 20  sensor_16  20631 

In [8]:
train_df.describe()

,unit_id,cycle,op1,op2,op3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.0,20631.00,20631.000000,20631.000000,20631.000000,2.063100e+04,...,20631.000000,20631.000000,20631.000000,20631.000000,2.063100e+04,20631.000000,20631.0,20631.0,20631.000000,20631.000000
mean,51.506568,108.807862,-0.000009,0.000002,100.0,518.67,642.680934,1590.523119,1408.933782,1.462000e+01,...,521.413470,2388.096152,8143.752722,8.442146,3.000000e-02,393.210654,2388.0,100.0,38.816271,23.289705
std,29.227633,68.880990,0.002187,0.000293,0.0,0.00,0.500053,6.131150,9.000605,5.329200e-15,...,0.737553,0.071919,19.076176,0.037505,3.469531e-18,1.548763,0.0,0.0,0.180746,0.108251
min,1.000000,1.000000,-0.008700,-0.000600,100.0,518.67,641.210000,1571.040000,1382.250000,1.462000e+01,...,518.690000,2387.880000,8099.940000,8.324900,3.000000e-02,388.000000,2388.0,100.0,38.140000,22.894200
25%,26.000000,52.000000,-0.001500,-0.000200,100.0,518.67,642.325000,1586.260000,1402.360000,1.462000e+01,...,520.960000,2388.040000,8133.245000,8.414900,3.000000e-02,392.000000,2388.0,100.0,38.700000,23.221800
50%,52.000000,104.000000,0.000000,0.000000,100.0,518.67,642.640000,1590.100000,1408.040000,1.462000e+01,...,521.480000,2388.090000,8140.540000,8.438900,3.000000e-02,393.000000,2388.0,100.0,38.830000,23.297900
75%,77.000000,156.000000,0.001500,0.000300,100.0,518.67,643.000000,1594.380000,1414.555000,1.462000e+01,...,521.950000,2388.140000,8148.310000,8.465600,3.000000e-02,394.000000,2388.0,100.0,38.950000,23.366800
max,100.000000,362.000000,0.008700,0.000600,100.0,518.67,644.530000,1616.910000,1441.490000,1.462000e+01,...,523.380000,2388.560000,8293.720000,8.584800,3.000000e-02,400.000000,2388.0,100.0,39.430000,23.618400


## Step 4 - Adding RUL column to training data

Since the engines run till failure, the last cycle of each engine = failure point.
So RUL = max cycle of that engine - current cycle

In [9]:
# find max cycle for each engine
max_cycles = train_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']

# merge it back
train_df = train_df.merge(max_cycles, on='unit_id')

# calculate RUL
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

# dont need max_cycle anymore
train_df.drop('max_cycle', axis=1, inplace=True)

train_df[['unit_id', 'cycle', 'RUL']].head(10)

,unit_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


In [10]:
# capping RUL at 125 because very high RUL values dont really help the model
# engine is basically healthy if RUL > 125 so we treat all those as 125
train_df['RUL'] = train_df['RUL'].clip(upper=125)

## Step 5 - Feature Selection

Some sensors have constant values throughout (variance = 0), those are useless for prediction so removing them.

In [11]:
variances = train_df.var()
zero_var = variances[variances == 0].index.tolist()
print('columns with no variance (dropping these):', zero_var)

# also dropping unit_id and cycle since they are not sensor features
drop_cols = zero_var + ['unit_id', 'cycle']

X_train = train_df.drop(columns=drop_cols + ['RUL'])
y_train = train_df['RUL']

print('features used:', X_train.columns.tolist())

columns with no variance (dropping these): ['op3', 'sensor_1', 'sensor_10', 'sensor_18', 'sensor_19']
features used: ['op1', 'op2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_20', 'sensor_21']


## Step 6 - Scaling

Using MinMaxScaler to bring all sensor values between 0 and 1 so no single sensor dominates.

In [12]:
# these are all feature columns except unit_id and cycle
scale_cols = cols[2:]

scaler = MinMaxScaler()
scaled_arr = scaler.fit_transform(train_df[scale_cols])
scaled_df = pd.DataFrame(scaled_arr, columns=scale_cols)

# removing zero variance cols from scaled data too
model_features = [c for c in scale_cols if c not in zero_var]
X_train_final = scaled_df[model_features]

print(X_train_final.shape)

(20631, 19)


## Step 7 - Preparing Test Data

For test data we only take the last cycle of each engine (since thats when prediction is made) and compare with actual RUL values.

In [13]:
# get last row of each engine in test
test_last = test_df.groupby('unit_id').last().reset_index()

# scale using same scaler fitted on train
X_test_scaled = scaler.transform(test_last[scale_cols])
X_test_df = pd.DataFrame(X_test_scaled, columns=scale_cols)
X_test_final = X_test_df[model_features]

# clip actual RUL same way as train
y_test = rul_df['RUL'].clip(upper=125)

print('X_test shape:', X_test_final.shape)
print('y_test shape:', y_test.shape)

X_test shape: (100, 19)
y_test shape: (100,)


## Step 8 - Training and Evaluating Each Model

Training each model one by one and checking train + test performance separately.

### Model 1 - Linear Regression

In [14]:
lr = LinearRegression()
lr.fit(X_train_final, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [15]:
# train performance
lr_train_pred = lr.predict(X_train_final)
print('--- Linear Regression Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, lr_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, lr_train_pred), 2))
print('R2   :', round(r2_score(y_train, lr_train_pred), 4))

--- Linear Regression Train Performance ---
RMSE : 21.47
MAE  : 17.56
R2   : 0.7346


In [16]:
# test performance
lr_test_pred = lr.predict(X_test_final)
print('--- Linear Regression Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, lr_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, lr_test_pred), 2))
print('R2   :', round(r2_score(y_test, lr_test_pred), 4))

--- Linear Regression Test Performance ---
RMSE : 20.83
MAE  : 16.57
R2   : 0.7298


### Model 2 - KNN

In [17]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_final, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [18]:
# train performance
knn_train_pred = knn.predict(X_train_final)
print('--- KNN Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, knn_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, knn_train_pred), 2))
print('R2   :', round(r2_score(y_train, knn_train_pred), 4))

--- KNN Train Performance ---
RMSE : 16.68
MAE  : 11.71
R2   : 0.8397


In [19]:
# test performance
knn_test_pred = knn.predict(X_test_final)
print('--- KNN Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, knn_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, knn_test_pred), 2))
print('R2   :', round(r2_score(y_test, knn_test_pred), 4))

--- KNN Test Performance ---
RMSE : 20.17
MAE  : 14.47
R2   : 0.7467


### Model 3 - Decision Tree

In [20]:
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train_final, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_lea

In [21]:
# train performance
dt_train_pred = dt.predict(X_train_final)
print('--- Decision Tree Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, dt_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, dt_train_pred), 2))
print('R2   :', round(r2_score(y_train, dt_train_pred), 4))

--- Decision Tree Train Performance ---
RMSE : 16.65
MAE  : 11.6
R2   : 0.8403


In [22]:
# test performance
dt_test_pred = dt.predict(X_test_final)
print('--- Decision Tree Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, dt_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, dt_test_pred), 2))
print('R2   :', round(r2_score(y_test, dt_test_pred), 4))

--- Decision Tree Test Performance ---
RMSE : 19.5
MAE  : 14.59
R2   : 0.7631


### Model 4 - Random Forest

In [23]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_final, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [24]:
# train performance
rf_train_pred = rf.predict(X_train_final)
print('--- Random Forest Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, rf_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, rf_train_pred), 2))
print('R2   :', round(r2_score(y_train, rf_train_pred), 4))

--- Random Forest Train Performance ---
RMSE : 15.32
MAE  : 11.11
R2   : 0.8649


In [25]:
# test performance
rf_test_pred = rf.predict(X_test_final)
print('--- Random Forest Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, rf_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, rf_test_pred), 2))
print('R2   :', round(r2_score(y_test, rf_test_pred), 4))

--- Random Forest Test Performance ---
RMSE : 17.22
MAE  : 12.2
R2   : 0.8154


### Model 5 - Bagging Regressor

In [26]:
bag = BaggingRegressor(n_estimators=50, random_state=42)
bag.fit(X_train_final, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",50
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [27]:
# train performance
bag_train_pred = bag.predict(X_train_final)
print('--- Bagging Regressor Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, bag_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, bag_train_pred), 2))
print('R2   :', round(r2_score(y_train, bag_train_pred), 4))

--- Bagging Regressor Train Performance ---
RMSE : 6.95
MAE  : 4.96
R2   : 0.9722


In [28]:
# test performance
bag_test_pred = bag.predict(X_test_final)
print('--- Bagging Regressor Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, bag_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, bag_test_pred), 2))
print('R2   :', round(r2_score(y_test, bag_test_pred), 4))

--- Bagging Regressor Test Performance ---
RMSE : 17.94
MAE  : 12.46
R2   : 0.7996


### Model 6 - Gradient Boosting

In [29]:
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gb.fit(X_train_final, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [30]:
# train performance
gb_train_pred = gb.predict(X_train_final)
print('--- Gradient Boosting Train Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_train, gb_train_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_train, gb_train_pred), 2))
print('R2   :', round(r2_score(y_train, gb_train_pred), 4))

--- Gradient Boosting Train Performance ---
RMSE : 16.23
MAE  : 11.72
R2   : 0.8484


In [31]:
# test performance
gb_test_pred = gb.predict(X_test_final)
print('--- Gradient Boosting Test Performance ---')
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, gb_test_pred)), 2))
print('MAE  :', round(mean_absolute_error(y_test, gb_test_pred), 2))
print('R2   :', round(r2_score(y_test, gb_test_pred), 4))

--- Gradient Boosting Test Performance ---
RMSE : 17.81
MAE  : 12.64
R2   : 0.8026


## Step 9 - Comparing all models together

In [32]:
# putting all test results in one table to compare easily
comparison = {
    'Model': ['Linear Regression', 'KNN', 'Decision Tree', 'Random Forest', 'Bagging', 'Gradient Boosting'],
    'Test RMSE': [
        round(np.sqrt(mean_squared_error(y_test, lr_test_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, knn_test_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, dt_test_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, rf_test_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, bag_test_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, gb_test_pred)), 2)
    ],
    'Test MAE': [
        round(mean_absolute_error(y_test, lr_test_pred), 2),
        round(mean_absolute_error(y_test, knn_test_pred), 2),
        round(mean_absolute_error(y_test, dt_test_pred), 2),
        round(mean_absolute_error(y_test, rf_test_pred), 2),
        round(mean_absolute_error(y_test, bag_test_pred), 2),
        round(mean_absolute_error(y_test, gb_test_pred), 2)
    ],
    'Test R2': [
        round(r2_score(y_test, lr_test_pred), 4),
        round(r2_score(y_test, knn_test_pred), 4),
        round(r2_score(y_test, dt_test_pred), 4),
        round(r2_score(y_test, rf_test_pred), 4),
        round(r2_score(y_test, bag_test_pred), 4),
        round(r2_score(y_test, gb_test_pred), 4)
    ]
}

comp_df = pd.DataFrame(comparison)
comp_df.sort_values('Test RMSE').reset_index(drop=True)

,Model,Test RMSE,Test MAE,Test R2
0,Random Forest,17.22,12.20,0.8154
1,Gradient Boosting,17.81,12.64,0.8026
2,Bagging,17.94,12.46,0.7996
3,Decision Tree,19.50,14.59,0.7631
4,KNN,20.17,14.47,0.7467
5,Linear Regression,20.83,16.57,0.7298


## Step 10 - Saving the Best Model

Based on test RMSE, saving the best performing model along with the scaler.

In [33]:
# saving random forest as final model (change this to whichever gave lowest test RMSE)
final_model = rf

with open('best_rul_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# saving scaler too because we need same scaling when predicting on new data
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# also saving the list of features the model was trained on
with open('model_features.pkl', 'wb') as f:
    pickle.dump(model_features, f)

print('saved: best_rul_model.pkl')
print('saved: scaler.pkl')
print('saved: model_features.pkl')

saved: best_rul_model.pkl
saved: scaler.pkl
saved: model_features.pkl


In [34]:
# quick test to make sure saved model loads and works correctly
with open('best_rul_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

with open('model_features.pkl', 'rb') as f:
    loaded_features = pickle.load(f)

# predict using loaded model
check_pred = loaded_model.predict(X_test_final)
check_rmse = round(np.sqrt(mean_squared_error(y_test, check_pred)), 2)
print('loaded model test RMSE:', check_rmse)
print('model loaded and working fine!')

loaded model test RMSE: 17.22
model loaded and working fine!
